# FinanceRAG-Analyst — Document Processing and Token-Aware Chunking

This notebook transforms the frozen SEC filing corpus into a deterministic,
provenance-preserving chunk collection for downstream retrieval evaluation.

## Responsibilities

- Load and verify the frozen 30-document corpus contract
- Conservatively normalize filing text
- Tokenize using the `BAAI/bge-small-en-v1.5` tokenizer
- Produce sentence-aware chunks within the dense model's input limit
- Preserve document and character-span provenance
- Validate chunk coverage, boundaries, duplication, and determinism
- Save versioned processed-document and chunk artifacts

## Exclusions

This notebook does not:

- download SEC filings
- build BM25 or FAISS indexes
- assign heuristic topic or relevance labels
- create evaluation questions
- tune retrieval configurations
- generate RAG answers

In [ ]:
%pip install -q \
    "pandas>=2.0,<3.0" \
    "pyarrow>=14.0,<20.0" \
    "transformers>=4.40,<5.0" \
    "tqdm>=4.66,<5.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import re
import sys
import unicodedata
from importlib.metadata import version
from pathlib import Path
from typing import Any

import pandas as pd
import pyarrow
import transformers
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)

DEPENDENCY_VERSIONS = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "transformers": transformers.__version__,
}

display(pd.Series(DEPENDENCY_VERSIONS, name="version").to_frame())

,version
python,3.12.13
platform,Linux-6.6.122+-x86_64-with-glibc2.35
pandas,2.2.2
pyarrow,18.1.0
transformers,4.57.6


In [ ]:
from transformers import AutoTokenizer

TOKENIZER_NAME = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME,
    use_fast=True,
)

test_text = "Apple reported net sales and operating income for the fiscal year."

encoded = tokenizer(
    test_text,
    add_special_tokens=True,
    truncation=False,
)

print(f"Tokenizer: {TOKENIZER_NAME}")
print(f"Tokenizer class: {tokenizer.__class__.__name__}")
print(f"Model maximum length: {tokenizer.model_max_length}")
print(f"Test tokens including special tokens: {len(encoded['input_ids'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizer: BAAI/bge-small-en-v1.5
Tokenizer class: BertTokenizerFast
Model maximum length: 512
Test tokens including special tokens: 14


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/finance-rag-analyst"
)

assert PROJECT_ROOT.exists(), (
    f"Project directory does not exist: {PROJECT_ROOT}"
)

print(f"Project root: {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/finance-rag-analyst


Frozen corpus and chunking configuration

In [ ]:
CORPUS_VERSION = "sec30_959ac4a68dbf"

RAW_CSV_PATH = (
    PROJECT_ROOT / "data/raw/sec_filings_raw.csv"
)

MANIFEST_CSV_PATH = (
    PROJECT_ROOT
    / "data/manifests"
    / f"corpus_manifest_{CORPUS_VERSION}.csv"
)

CORPUS_METADATA_PATH = (
    PROJECT_ROOT
    / "data/manifests"
    / f"corpus_metadata_{CORPUS_VERSION}.json"
)

TOKENIZER_NAME = "BAAI/bge-small-en-v1.5"

PROCESSING_VERSION = "conservative_text_v1"
CHUNKING_VERSION = "bge_sentence_480_80_v1"

MAX_CHUNK_TOKENS = 480
TARGET_OVERLAP_TOKENS = 80
MIN_TARGET_TOKENS = 100

REQUIRED_INPUT_PATHS = {
    "raw filing CSV": RAW_CSV_PATH,
    "frozen corpus manifest": MANIFEST_CSV_PATH,
    "corpus metadata": CORPUS_METADATA_PATH,
}

for label, path in REQUIRED_INPUT_PATHS.items():
    assert path.exists(), f"Missing {label}: {path}"
    print(f"Found {label}: {path.relative_to(PROJECT_ROOT)}")

Found raw filing CSV: data/raw/sec_filings_raw.csv
Found frozen corpus manifest: data/manifests/corpus_manifest_sec30_959ac4a68dbf.csv
Found corpus metadata: data/manifests/corpus_metadata_sec30_959ac4a68dbf.json


Configuration record

In [ ]:
PIPELINE_CONFIG = {
    "corpus_version": CORPUS_VERSION,
    "processing_version": PROCESSING_VERSION,
    "chunking_version": CHUNKING_VERSION,
    "tokenizer_name": TOKENIZER_NAME,
    "tokenizer_class": tokenizer.__class__.__name__,
    "tokenizer_model_max_length": int(tokenizer.model_max_length),
    "max_chunk_tokens_including_special_tokens": MAX_CHUNK_TOKENS,
    "target_overlap_tokens": TARGET_OVERLAP_TOKENS,
    "minimum_target_tokens": MIN_TARGET_TOKENS,
    "dependency_versions": DEPENDENCY_VERSIONS,
}

assert MAX_CHUNK_TOKENS < tokenizer.model_max_length
assert MIN_TARGET_TOKENS < MAX_CHUNK_TOKENS
assert TARGET_OVERLAP_TOKENS < MAX_CHUNK_TOKENS

display(pd.Series(PIPELINE_CONFIG, name="value").to_frame())

,value
corpus_version,sec30_959ac4a68dbf
processing_version,conservative_text_v1
chunking_version,bge_sentence_480_80_v1
tokenizer_name,BAAI/bge-small-en-v1.5
tokenizer_class,BertTokenizerFast
tokenizer_model_max_length,512
max_chunk_tokens_including_special_tokens,480
target_overlap_tokens,80
minimum_target_tokens,100
dependency_versions,"{'python': '3.12.13', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'pandas': '2.2.2', 'pyarrow': '18.1.0', 'transformers': '4.57.6'}"


Load the frozen corpus artifacts

In [ ]:
raw_df = pd.read_csv(
    RAW_CSV_PATH,
    low_memory=False,
)

manifest_df = pd.read_csv(
    MANIFEST_CSV_PATH,
    dtype={
        "cik": "string",
        "accession_number": "string",
    },
)

with CORPUS_METADATA_PATH.open("r", encoding="utf-8") as file:
    corpus_metadata = json.load(file)

print(f"Raw filing rows: {len(raw_df):,}")
print(f"Manifest rows: {len(manifest_df):,}")
print(f"Metadata corpus version: {corpus_metadata.get('corpus_version')}")

Raw filing rows: 30
Manifest rows: 30
Metadata corpus version: sec30_959ac4a68dbf


Inspect the schemas

In [ ]:
def summarize_dataframe(
    dataframe: pd.DataFrame,
    dataframe_name: str,
) -> pd.DataFrame:
    summary_rows = []

    for column in dataframe.columns:
        series = dataframe[column]

        summary_rows.append(
            {
                "dataframe": dataframe_name,
                "column": column,
                "dtype": str(series.dtype),
                "non_null": int(series.notna().sum()),
                "null_count": int(series.isna().sum()),
                "unique_count": int(series.nunique(dropna=True)),
            }
        )

    return pd.DataFrame(summary_rows)


schema_summary = pd.concat(
    [
        summarize_dataframe(raw_df, "raw_df"),
        summarize_dataframe(manifest_df, "manifest_df"),
    ],
    ignore_index=True,
)

display(schema_summary)

,dataframe,column,dtype,non_null,null_count,unique_count
0,raw_df,ticker,object,30,0,5
1,raw_df,company,object,30,0,5
2,raw_df,cik,int64,30,0,5
3,raw_df,form_type,object,30,0,2
4,raw_df,filing_date,object,30,0,27
5,raw_df,report_date,object,30,0,20
6,raw_df,accession_number,object,30,0,30
7,raw_df,primary_document,object,30,0,30
8,raw_df,source_url,object,30,0,30
9,raw_df,text,object,30,0,30


Display safe sample metadata

In [ ]:
RAW_PREVIEW_COLUMNS = [
    column
    for column in [
        "ticker",
        "company",
        "company_name",
        "cik",
        "form_type",
        "filing_date",
        "report_date",
        "accession_number",
        "primary_document",
        "source_url",
        "text",
    ]
    if column in raw_df.columns
]

MANIFEST_PREVIEW_COLUMNS = [
    column
    for column in [
        "document_id",
        "ticker",
        "company",
        "company_name",
        "cik",
        "form_type",
        "filing_date",
        "report_date",
        "accession_number",
        "primary_document",
        "source_url",
        "content_sha256",
        "raw_char_count",
    ]
    if column in manifest_df.columns
]

raw_preview = raw_df[RAW_PREVIEW_COLUMNS].head(3).copy()

if "text" in raw_preview.columns:
    raw_preview["text"] = raw_preview["text"].map(
        lambda value: (
            f"{str(value)[:300]}..."
            if pd.notna(value)
            else value
        )
    )

print("Raw filing sample:")
display(raw_preview)

print("Frozen manifest sample:")
display(manifest_df[MANIFEST_PREVIEW_COLUMNS].head(3))

Raw filing sample:


,ticker,company,cik,form_type,filing_date,report_date,accession_number,primary_document,source_url,text
0,NVDA,NVIDIA CORP,1045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm,"nvda-20260125 Table of Contents UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ____________________________________________________..."
1,NVDA,NVIDIA CORP,1045810,10-K,2025-02-26,2025-01-26,0001045810-25-000023,nvda-20250126.htm,https://www.sec.gov/Archives/edgar/data/1045810/000104581025000023/nvda-20250126.htm,"nvda-20250126 Table of Contents UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ____________________________________________________..."
2,NVDA,NVIDIA CORP,1045810,10-K,2024-02-21,2024-01-28,0001045810-24-000029,nvda-20240128.htm,https://www.sec.gov/Archives/edgar/data/1045810/000104581024000029/nvda-20240128.htm,"nvda-20240128 Table of Contents UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ____________________________________________________..."


Frozen manifest sample:


,document_id,ticker,company,cik,form_type,filing_date,report_date,accession_number,primary_document,source_url,content_sha256,raw_char_count
0,aapl_10k_000032019323000106,AAPL,Apple Inc.,0000320193,10-K,2023-11-03,2023-09-30,0000320193-23-000106,aapl-20230930.htm,https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm,06fffabc25f99c8e9b02d81530df9df87fa476244c51d8b4114dcba6022eaacd,200598
1,aapl_10k_000032019324000123,AAPL,Apple Inc.,0000320193,10-K,2024-11-01,2024-09-28,0000320193-24-000123,aapl-20240928.htm,https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.htm,48dadd62ed7cfdcdf2f0818762cf9eba9021b9db46afbae095d3ef092bb32661,204321
2,aapl_10k_000032019325000079,AAPL,Apple Inc.,0000320193,10-K,2025-10-31,2025-09-27,0000320193-25-000079,aapl-20250927.htm,https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm,79983dc36aac906b94096f41d4c19f224bbc91ac3959e3063e9aa99a2c95b4ad,206727


In [ ]:
print(
    json.dumps(
        corpus_metadata,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "corpus_sha256": "959ac4a68dbfaf74dbd667bf9da8520d4727516c26776e46d9f05ba17cda1ff9",
  "corpus_version": "sec30_959ac4a68dbf",
  "document_count": 30,
  "documents_by_ticker_and_form": {
    "AAPL_10-K": 3,
    "AAPL_10-Q": 3,
    "AMZN_10-K": 3,
    "AMZN_10-Q": 3,
    "GOOGL_10-K": 3,
    "GOOGL_10-Q": 3,
    "MSFT_10-K": 3,
    "MSFT_10-Q": 3,
    "NVDA_10-K": 3,
    "NVDA_10-Q": 3
  },
  "fingerprint_fields": [
    "document_id",
    "accession_number",
    "content_sha256"
  ],
  "forms": [
    "10-K",
    "10-Q"
  ],
  "manifest_filename": "corpus_manifest_sec30_959ac4a68dbf.csv",
  "source_artifact": "data/raw/sec_filings_raw.csv",
  "ticker_count": 5,
  "tickers": [
    "AAPL",
    "AMZN",
    "GOOGL",
    "MSFT",
    "NVDA"
  ]
}


Frozen-input integrity gate

In [ ]:
def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def canonical_cik(value: Any) -> str:
    value = str(value).strip()

    if value.endswith(".0"):
        value = value[:-2]

    if not value.isdigit():
        raise ValueError(f"Invalid CIK: {value!r}")

    return value.zfill(10)


assert corpus_metadata["corpus_version"] == CORPUS_VERSION
assert corpus_metadata["document_count"] == 30

assert len(raw_df) == 30
assert len(manifest_df) == 30

assert raw_df["accession_number"].is_unique
assert manifest_df["accession_number"].is_unique
assert manifest_df["document_id"].is_unique

raw_contract_df = raw_df.copy()
manifest_contract_df = manifest_df.copy()

raw_contract_df["accession_number"] = (
    raw_contract_df["accession_number"].astype(str).str.strip()
)

manifest_contract_df["accession_number"] = (
    manifest_contract_df["accession_number"].astype(str).str.strip()
)

raw_contract_df["canonical_cik"] = (
    raw_contract_df["cik"].map(canonical_cik)
)

manifest_contract_df["canonical_cik"] = (
    manifest_contract_df["cik"].map(canonical_cik)
)

raw_contract_df["computed_content_sha256"] = (
    raw_contract_df["text"].map(sha256_text)
)

contract_check_df = manifest_contract_df[
    [
        "document_id",
        "accession_number",
        "canonical_cik",
        "content_sha256",
    ]
].merge(
    raw_contract_df[
        [
            "accession_number",
            "canonical_cik",
            "computed_content_sha256",
        ]
    ],
    on="accession_number",
    how="outer",
    validate="one_to_one",
    suffixes=("_manifest", "_raw"),
    indicator=True,
)

assert len(contract_check_df) == 30
assert contract_check_df["_merge"].eq("both").all()

assert (
    contract_check_df["canonical_cik_manifest"]
    == contract_check_df["canonical_cik_raw"]
).all()

assert (
    contract_check_df["content_sha256"]
    == contract_check_df["computed_content_sha256"]
).all()

print(
    "Frozen input contract passed: "
    "30/30 documents matched by accession, CIK, and raw-text SHA-256."
)

Frozen input contract passed: 30/30 documents matched by accession, CIK, and raw-text SHA-256.


canonical document table

In [ ]:
CANONICAL_METADATA_COLUMNS = [
    "document_id",
    "ticker",
    "company",
    "cik",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "primary_document",
    "source_url",
    "content_sha256",
    "raw_char_count",
    "raw_word_count",
]

documents_df = manifest_df[
    CANONICAL_METADATA_COLUMNS
].merge(
    raw_contract_df[
        [
            "accession_number",
            "text",
        ]
    ],
    on="accession_number",
    how="left",
    validate="one_to_one",
)

documents_df = documents_df.rename(
    columns={
        "text": "raw_text",
        "content_sha256": "raw_content_sha256",
    }
)

assert len(documents_df) == 30
assert documents_df["document_id"].is_unique
assert documents_df["raw_text"].notna().all()

print(f"Canonical documents created: {len(documents_df)}")

display(
    documents_df[
        [
            "document_id",
            "ticker",
            "form_type",
            "filing_date",
            "raw_char_count",
        ]
    ].head()
)

Canonical documents created: 30


,document_id,ticker,form_type,filing_date,raw_char_count
0,aapl_10k_000032019323000106,AAPL,10-K,2023-11-03,200598
1,aapl_10k_000032019324000123,AAPL,10-K,2024-11-01,204321
2,aapl_10k_000032019325000079,AAPL,10-K,2025-10-31,206727
3,aapl_10q_000032019325000073,AAPL,10-Q,2025-08-01,59899
4,aapl_10q_000032019326000006,AAPL,10-Q,2026-01-30,56972


Conservative text normalization

In [ ]:
UNICODE_SPACE_PATTERN = re.compile(
    r"[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]"
)

HORIZONTAL_WHITESPACE_PATTERN = re.compile(r"[ \t\f\v]+")
EXCESS_BLANK_LINES_PATTERN = re.compile(r"\n{3,}")


def normalize_filing_text(raw_text: str) -> str:
    text = unicodedata.normalize("NFC", raw_text)

    # Normalize line endings and remove null bytes.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\x00", "")

    # Convert unusual Unicode spaces to ordinary spaces.
    text = UNICODE_SPACE_PATTERN.sub(" ", text)

    # Normalize horizontal whitespace without flattening line structure.
    text = HORIZONTAL_WHITESPACE_PATTERN.sub(" ", text)

    # Remove spaces surrounding line breaks.
    text = re.sub(r" *\n *", "\n", text)

    # Retain paragraph boundaries while removing pathological blank space.
    text = EXCESS_BLANK_LINES_PATTERN.sub("\n\n", text)

    return text.strip()

processing and record hashes

In [ ]:
#Apply conservative normalization

processed_documents_df = documents_df.copy()

processed_documents_df["processed_text"] = (
    processed_documents_df["raw_text"].map(normalize_filing_text)
)

processed_documents_df["processed_content_sha256"] = (
    processed_documents_df["processed_text"].map(sha256_text)
)

processed_documents_df["processed_char_count"] = (
    processed_documents_df["processed_text"].str.len()
)

processed_documents_df["processing_version"] = PROCESSING_VERSION
processed_documents_df["corpus_version"] = CORPUS_VERSION

# Minimal safety checks.
assert len(processed_documents_df) == 30
assert processed_documents_df["processed_text"].notna().all()
assert processed_documents_df["processed_text"].str.len().gt(0).all()

raw_character_count = int(
    processed_documents_df["raw_text"].str.len().sum()
)

processed_character_count = int(
    processed_documents_df["processed_char_count"].sum()
)

changed_document_count = int(
    processed_documents_df["raw_text"]
    .ne(processed_documents_df["processed_text"])
    .sum()
)

print(f"Processed documents: {len(processed_documents_df)}")
print(
    "Documents changed by conservative normalization: "
    f"{changed_document_count}"
)
print(
    f"Corpus characters: {raw_character_count:,} raw → "
    f"{processed_character_count:,} processed"
)

Processed documents: 30
Documents changed by conservative normalization: 0
Corpus characters: 6,735,260 raw → 6,735,260 processed


Detect sentence-like spans with exact offsets

In [ ]:
# Cell 15 — Sentence-like span segmentation with exact offsets

SENTENCE_BOUNDARY_PATTERN = re.compile(
    r"""
    (?:
        (?<=[.!?])          # sentence-ending punctuation
        (?=\s+[A-Z0-9(])    # likely beginning of the next sentence
    )
    |
    (?:
        \n{2,}              # paragraph boundary
    )
    """,
    flags=re.VERBOSE,
)


def segment_sentence_like_spans(
    text: str,
) -> list[dict[str, Any]]:
    """
    Divide processed filing text into deterministic sentence-like spans.

    Every returned span preserves exact character offsets into `text`.
    No text is rewritten, discarded, or duplicated.
    """
    if not text:
        return []

    boundary_positions = {
        match.start()
        for match in SENTENCE_BOUNDARY_PATTERN.finditer(text)
    }

    boundary_positions.add(len(text))

    spans = []
    start = 0

    for boundary in sorted(boundary_positions):
        if boundary <= start:
            continue

        raw_span = text[start:boundary]

        leading_whitespace = len(raw_span) - len(raw_span.lstrip())
        trailing_boundary = boundary - (
            len(raw_span) - len(raw_span.rstrip())
        )

        span_start = start + leading_whitespace
        span_end = trailing_boundary

        if span_start < span_end:
            span_text = text[span_start:span_end]

            spans.append(
                {
                    "span_index": len(spans),
                    "start_char": span_start,
                    "end_char": span_end,
                    "text": span_text,
                }
            )

        start = boundary

    return spans

Validate segmentation on one filing

In [ ]:
# Cell 16 — One-document segmentation sanity check

sample_document = processed_documents_df.iloc[0]

sample_text = sample_document["processed_text"]
sample_spans = segment_sentence_like_spans(sample_text)

assert sample_spans, "No spans were produced."

for expected_index, span in enumerate(sample_spans):
    assert span["span_index"] == expected_index
    assert 0 <= span["start_char"] < span["end_char"] <= len(sample_text)

    assert (
        sample_text[
            span["start_char"]:span["end_char"]
        ]
        == span["text"]
    )

for previous_span, current_span in zip(
    sample_spans,
    sample_spans[1:],
):
    assert previous_span["end_char"] <= current_span["start_char"]

print(f"Document: {sample_document['document_id']}")
print(f"Sentence-like spans detected: {len(sample_spans):,}")
print("Exact span offsets verified: true")

Document: aapl_10k_000032019323000106
Sentence-like spans detected: 940
Exact span offsets verified: true


Split oversized spans using BGE token offsets

In [ ]:
# Cell 17 — Split oversized spans using tokenizer offsets

SPECIAL_TOKEN_COUNT = tokenizer.num_special_tokens_to_add(pair=False)
MAX_CONTENT_TOKENS = MAX_CHUNK_TOKENS - SPECIAL_TOKEN_COUNT

assert SPECIAL_TOKEN_COUNT == 2
assert MAX_CONTENT_TOKENS == 478


def count_tokens(text: str) -> int:
    """Count BGE tokens, including special tokens."""
    return len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )


def split_oversized_span(
    span: dict[str, Any],
) -> list[dict[str, Any]]:
    """
    Split one sentence-like span only when it exceeds the configured
    token ceiling.

    Character offsets remain relative to the complete processed document.
    """
    span_text = span["text"]

    encoding = tokenizer(
        span_text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
    )

    token_offsets = encoding["offset_mapping"]

    if len(token_offsets) <= MAX_CONTENT_TOKENS:
        return [
            {
                **span,
                "source_span_index": span["span_index"],
                "split_index": 0,
                "was_split": False,
                "token_count": len(token_offsets) + SPECIAL_TOKEN_COUNT,
            }
        ]

    pieces = []

    for split_index, token_start in enumerate(
        range(0, len(token_offsets), MAX_CONTENT_TOKENS)
    ):
        token_end = min(
            token_start + MAX_CONTENT_TOKENS,
            len(token_offsets),
        )

        # Include any whitespace between adjacent token windows in the
        # preceding piece so the original span remains fully covered.
        local_start = (
            0
            if token_start == 0
            else token_offsets[token_start][0]
        )

        local_end = (
            len(span_text)
            if token_end == len(token_offsets)
            else token_offsets[token_end][0]
        )

        piece_text = span_text[local_start:local_end]

        piece = {
            "span_index": None,
            "source_span_index": span["span_index"],
            "split_index": split_index,
            "was_split": True,
            "start_char": span["start_char"] + local_start,
            "end_char": span["start_char"] + local_end,
            "text": piece_text,
            "token_count": count_tokens(piece_text),
        }

        assert piece["token_count"] <= MAX_CHUNK_TOKENS
        pieces.append(piece)

    return pieces

Create token-bounded atomic spans for all documents

In [ ]:
# Cell 18 — Build token-bounded atomic spans

atomic_spans_by_document = {}

total_sentence_spans = 0
total_atomic_spans = 0
oversized_source_spans = 0

for row in tqdm(
    processed_documents_df.itertuples(index=False),
    total=len(processed_documents_df),
    desc="Creating token-bounded spans",
):
    sentence_spans = segment_sentence_like_spans(
        row.processed_text
    )

    atomic_spans = []

    for sentence_span in sentence_spans:
        pieces = split_oversized_span(sentence_span)

        if pieces[0]["was_split"]:
            oversized_source_spans += 1

        atomic_spans.extend(pieces)

    for atomic_index, atomic_span in enumerate(atomic_spans):
        atomic_span["span_index"] = atomic_index

        assert (
            row.processed_text[
                atomic_span["start_char"]:
                atomic_span["end_char"]
            ]
            == atomic_span["text"]
        )

        assert atomic_span["token_count"] <= MAX_CHUNK_TOKENS

    atomic_spans_by_document[row.document_id] = atomic_spans

    total_sentence_spans += len(sentence_spans)
    total_atomic_spans += len(atomic_spans)

assert len(atomic_spans_by_document) == len(processed_documents_df)

print(f"Documents segmented: {len(atomic_spans_by_document)}")
print(f"Original sentence-like spans: {total_sentence_spans:,}")
print(f"Oversized source spans split: {oversized_source_spans:,}")
print(f"Token-bounded atomic spans: {total_atomic_spans:,}")
print(f"Maximum permitted span tokens: {MAX_CHUNK_TOKENS}")

Creating token-bounded spans:   0%|          | 0/30 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (695 > 512). Running this sequence through the model will result in indexing errors


Documents segmented: 30
Original sentence-like spans: 31,231
Oversized source spans split: 150
Token-bounded atomic spans: 31,402
Maximum permitted span tokens: 480


Pack atomic spans into token-bounded chunks

In [ ]:
MINIMUM_TARGET_TOKENS = 100

In [ ]:
# Cell 19 — Pack atomic spans without redundant overlap-only chunks

def make_chunk(
    document_id: str,
    chunk_index: int,
    processed_text: str,
    atomic_spans: list[dict[str, Any]],
    start_span_index: int,
    end_span_index: int,
) -> dict[str, Any]:
    """Create one exact-substring chunk from an inclusive atomic-span range."""
    start_char = atomic_spans[start_span_index]["start_char"]
    end_char = atomic_spans[end_span_index]["end_char"]

    chunk_text = processed_text[start_char:end_char]
    token_count = count_tokens(chunk_text)

    assert chunk_text
    assert token_count <= MAX_CHUNK_TOKENS
    assert processed_text[start_char:end_char] == chunk_text

    return {
        "document_id": document_id,
        "chunk_index": chunk_index,
        "start_span_index": start_span_index,
        "end_span_index": end_span_index,
        "start_char": start_char,
        "end_char": end_char,
        "text": chunk_text,
        "token_count": token_count,
    }


def build_document_chunks(
    document_id: str,
    processed_text: str,
    atomic_spans: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    Greedily pack adjacent atomic spans into chunks of at most 480 BGE tokens.

    Subsequent chunks retain atomic-span-aligned overlap where possible, but a
    candidate containing only spans already covered by the preceding chunk is
    skipped rather than emitted as a redundant short chunk.
    """
    if not atomic_spans:
        return []

    chunks = []
    start_span_index = 0

    while start_span_index < len(atomic_spans):
        accepted_end_span_index = None

        for candidate_end_span_index in range(
            start_span_index,
            len(atomic_spans),
        ):
            candidate_start_char = atomic_spans[
                start_span_index
            ]["start_char"]

            candidate_end_char = atomic_spans[
                candidate_end_span_index
            ]["end_char"]

            candidate_text = processed_text[
                candidate_start_char:candidate_end_char
            ]

            if count_tokens(candidate_text) > MAX_CHUNK_TOKENS:
                break

            accepted_end_span_index = candidate_end_span_index

        # Every atomic span was previously verified to fit by itself.
        assert accepted_end_span_index is not None

        # If overlap prevents the candidate from reaching any new span, the
        # candidate would add no retrieval coverage. Drop that overlap and
        # resume from the first span not already covered.
        if (
            chunks
            and accepted_end_span_index
            <= chunks[-1]["end_span_index"]
        ):
            start_span_index = (
                chunks[-1]["end_span_index"] + 1
            )
            continue

        chunks.append(
            make_chunk(
                document_id=document_id,
                chunk_index=len(chunks),
                processed_text=processed_text,
                atomic_spans=atomic_spans,
                start_span_index=start_span_index,
                end_span_index=accepted_end_span_index,
            )
        )

        if accepted_end_span_index == len(atomic_spans) - 1:
            break

        # A single-span chunk cannot retain that same span as overlap without
        # preventing forward progress.
        if accepted_end_span_index == start_span_index:
            start_span_index += 1
            continue

        # Retain a suffix of complete atomic spans as overlap.
        next_start_span_index = accepted_end_span_index

        while next_start_span_index > start_span_index + 1:
            overlap_start_char = atomic_spans[
                next_start_span_index
            ]["start_char"]

            overlap_end_char = atomic_spans[
                accepted_end_span_index
            ]["end_char"]

            overlap_text = processed_text[
                overlap_start_char:overlap_end_char
            ]

            if count_tokens(overlap_text) >= TARGET_OVERLAP_TOKENS:
                break

            next_start_span_index -= 1

        assert next_start_span_index > start_span_index
        start_span_index = next_start_span_index

    # Repair an undersized final residual when backward expansion is possible.
    if (
        len(chunks) >= 2
        and chunks[-1]["token_count"] < MINIMUM_TARGET_TOKENS
    ):
        previous_chunk = chunks[-2]
        final_chunk = chunks[-1]

        repaired_start = final_chunk["start_span_index"]

        while repaired_start > previous_chunk["start_span_index"]:
            candidate_start = repaired_start - 1

            candidate_text = processed_text[
                atomic_spans[candidate_start]["start_char"]:
                atomic_spans[final_chunk["end_span_index"]]["end_char"]
            ]

            if count_tokens(candidate_text) > MAX_CHUNK_TOKENS:
                break

            repaired_start = candidate_start

            if count_tokens(candidate_text) >= MINIMUM_TARGET_TOKENS:
                break

        chunks[-1] = make_chunk(
            document_id=document_id,
            chunk_index=len(chunks) - 1,
            processed_text=processed_text,
            atomic_spans=atomic_spans,
            start_span_index=repaired_start,
            end_span_index=final_chunk["end_span_index"],
        )

    for chunk_index, chunk in enumerate(chunks):
        chunk["chunk_index"] = chunk_index

    return chunks

Build chunks for the frozen corpus

In [ ]:
# Cell 20 — Build and validate retrieval chunks

chunks_by_document = {}
all_chunks = []

document_text_lookup = (
    processed_documents_df
    .set_index("document_id")["processed_text"]
    .to_dict()
)

for document_id, atomic_spans in tqdm(
    atomic_spans_by_document.items(),
    total=len(atomic_spans_by_document),
    desc="Building retrieval chunks",
):
    document_chunks = build_document_chunks(
        document_id=document_id,
        processed_text=document_text_lookup[document_id],
        atomic_spans=atomic_spans,
    )

    assert document_chunks, (
        f"No chunks created for document: {document_id}"
    )

    # Every new chunk must extend coverage beyond the preceding chunk.
    assert all(
        current_chunk["end_span_index"]
        > previous_chunk["end_span_index"]
        for previous_chunk, current_chunk in zip(
            document_chunks,
            document_chunks[1:],
        )
    )

    # Every atomic span must be covered by at least one retrieval chunk.
    covered_span_indices = {
        span_index
        for chunk in document_chunks
        for span_index in range(
            chunk["start_span_index"],
            chunk["end_span_index"] + 1,
        )
    }

    assert covered_span_indices == set(range(len(atomic_spans)))

    chunks_by_document[document_id] = document_chunks
    all_chunks.extend(document_chunks)

chunk_token_counts = [
    chunk["token_count"]
    for chunk in all_chunks
]

undersized_chunks = [
    chunk
    for chunk in all_chunks
    if chunk["token_count"] < MINIMUM_TARGET_TOKENS
]

assert len(chunks_by_document) == 30
assert max(chunk_token_counts) <= MAX_CHUNK_TOKENS

assert all(
    chunk["text"]
    == document_text_lookup[chunk["document_id"]][
        chunk["start_char"]:chunk["end_char"]
    ]
    for chunk in all_chunks
)

print(f"Documents chunked: {len(chunks_by_document)}")
print(f"Retrieval chunks created: {len(all_chunks):,}")
print(
    "Chunk-token range: "
    f"{min(chunk_token_counts):,}–{max(chunk_token_counts):,}"
)
print(
    "Mean chunk tokens: "
    f"{sum(chunk_token_counts) / len(chunk_token_counts):.1f}"
)
print(
    "Chunks below minimum target: "
    f"{len(undersized_chunks):,}"
)

Building retrieval chunks:   0%|          | 0/30 [00:00<?, ?it/s]

Documents chunked: 30
Retrieval chunks created: 4,174
Chunk-token range: 17–480
Mean chunk tokens: 434.7
Chunks below minimum target: 19


Inspect the undersized exceptions

In [ ]:
# Cell 21 — Inspect remaining undersized chunk exceptions

undersized_chunk_records = []

for chunk in all_chunks:
    if chunk["token_count"] >= MINIMUM_TARGET_TOKENS:
        continue

    document_chunks = chunks_by_document[chunk["document_id"]]

    undersized_chunk_records.append(
        {
            "document_id": chunk["document_id"],
            "chunk_index": chunk["chunk_index"],
            "token_count": chunk["token_count"],
            "span_count": (
                chunk["end_span_index"]
                - chunk["start_span_index"]
                + 1
            ),
            "is_first": chunk["chunk_index"] == 0,
            "is_final": (
                chunk["chunk_index"]
                == len(document_chunks) - 1
            ),
            "text_preview": (
                " ".join(chunk["text"].split())[:120]
            ),
        }
    )

undersized_chunks_df = (
    pd.DataFrame(undersized_chunk_records)
    .sort_values(
        ["token_count", "document_id", "chunk_index"]
    )
    .reset_index(drop=True)
)

assert len(undersized_chunks_df) == 19

print(
    f"Remaining undersized chunks: "
    f"{len(undersized_chunks_df):,}/{len(all_chunks):,} "
    f"({100 * len(undersized_chunks_df) / len(all_chunks):.2f}%)"
)
print(
    "Final-document residuals: "
    f"{int(undersized_chunks_df['is_final'].sum())}"
)
print(
    "Internal undersized chunks: "
    f"{int((~undersized_chunks_df['is_final']).sum())}"
)

display(undersized_chunks_df)

Remaining undersized chunks: 19/4,174 (0.46%)
Final-document residuals: 0
Internal undersized chunks: 19


,document_id,chunk_index,token_count,span_count,is_first,is_final,text_preview
0,googl_10q_000165204425000062,13,17,1,False,False,",033 $ 300,753 8 Table of Contents Alphabet Inc."
1,aapl_10q_000032019326000013,6,19,1,False,False,Apple Inc. | Q2 2026 Form 10-Q | 3 Apple Inc.
2,aapl_10q_000032019326000013,8,19,1,False,False,Apple Inc. | Q2 2026 Form 10-Q | 4 Apple Inc.
3,aapl_10q_000032019325000073,7,22,2,False,False,Financial Statements. Apple Inc. | Q3 2025 Form 10-Q | 4 Apple Inc.
4,googl_10q_000165204426000048,11,22,3,False,False,"$ 478,746 See accompanying notes. 7 Table of Contents Alphabet Inc. Alphabet Inc."
5,msft_10k_000095017023035122,116,28,1,False,False,",542 Total liabilities and stockholders’ equity $ 411,976 $ 364,840 Refer to accompanying notes."
6,msft_10q_000119312526027207,6,33,1,False,False,"390,875 343,479 Total liabilities and stockholders’ equity $ 665,302 $ 619,003 Refer to accompanying notes."
7,googl_10k_000165204424000022,124,36,4,False,False,"76,534 $ ( 4,402 ) $ 211,247 $ 283,379 See accompanying notes. 54. Table of Contents Alphabet Inc. Alphabet Inc."
8,msft_10k_000095017024087843,120,36,1,False,False,"Total stockholders’ equity 268,477 206,223 Total liabilities and stockholders’ equity $ 512,163 $ 411,976 Refer to accom"
9,msft_10k_000095017025100235,103,38,1,False,False,"Total stockholders’ equity 343,479 268,477 Total liabilities and stockholders’ equity $ 619,003 $ 512,163 Refer to accom"


Create the canonical chunk DataFrame

In [ ]:
processed_documents_df.columns.tolist()

['document_id',
 'ticker',
 'company',
 'cik',
 'form_type',
 'filing_date',
 'report_date',
 'accession_number',
 'primary_document',
 'source_url',
 'raw_content_sha256',
 'raw_char_count',
 'raw_word_count',
 'raw_text',
 'processed_text',
 'processed_content_sha256',
 'processed_char_count',
 'processing_version',
 'corpus_version']

In [ ]:
# Cell 22 — Create canonical retrieval-chunk dataset

document_metadata_columns = [
    "document_id",
    "ticker",
    "company",
    "cik",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "primary_document",
    "source_url",
    "raw_content_sha256",
    "processed_content_sha256",
    "processing_version",
    "corpus_version",
]

# All requested metadata columns must exist in the processed document table.
missing_metadata_columns = sorted(
    set(document_metadata_columns)
    - set(processed_documents_df.columns)
)

assert not missing_metadata_columns, (
    "Missing document metadata columns: "
    f"{missing_metadata_columns}"
)

document_metadata_df = (
    processed_documents_df[document_metadata_columns]
    .drop_duplicates(subset=["document_id"])
    .copy()
)

assert len(document_metadata_df) == 30
assert document_metadata_df["document_id"].is_unique

chunk_records = []

for chunk in all_chunks:
    chunk_id = (
        f"{chunk['document_id']}__chunk_"
        f"{chunk['chunk_index']:04d}"
    )

    chunk_records.append(
        {
            "chunk_id": chunk_id,
            "document_id": chunk["document_id"],
            "chunk_index": chunk["chunk_index"],
            "start_span_index": chunk["start_span_index"],
            "end_span_index": chunk["end_span_index"],
            "start_char": chunk["start_char"],
            "end_char": chunk["end_char"],
            "token_count": chunk["token_count"],
            "below_minimum_target": (
                chunk["token_count"] < MINIMUM_TARGET_TOKENS
            ),
            "text": chunk["text"],
        }
    )

chunks_df = (
    pd.DataFrame(chunk_records)
    .merge(
        document_metadata_df,
        on="document_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        ["document_id", "chunk_index"]
    )
    .reset_index(drop=True)
)

# Validate row identity and source-document representation.
assert len(chunks_df) == len(all_chunks) == 4_174
assert chunks_df["chunk_id"].is_unique
assert chunks_df["document_id"].nunique() == 30

# Validate chunk content and token constraints.
assert chunks_df["text"].notna().all()
assert chunks_df["text"].str.len().gt(0).all()
assert chunks_df["token_count"].le(MAX_CHUNK_TOKENS).all()
assert chunks_df["below_minimum_target"].sum() == 19

# Validate successful metadata attachment.
assert chunks_df["ticker"].notna().all()
assert chunks_df["company"].notna().all()
assert chunks_df["form_type"].notna().all()
assert chunks_df["filing_date"].notna().all()
assert chunks_df["source_url"].notna().all()
assert chunks_df["processed_content_sha256"].notna().all()

# Chunk indices must be contiguous within every document.
expected_chunk_indices = (
    chunks_df
    .groupby("document_id", sort=False)["chunk_index"]
    .apply(
        lambda values: (
            values.tolist()
            == list(range(len(values)))
        )
    )
)

assert expected_chunk_indices.all()

print(f"Canonical chunk rows: {len(chunks_df):,}")
print(f"Unique chunk IDs: {chunks_df['chunk_id'].nunique():,}")
print(
    "Documents represented: "
    f"{chunks_df['document_id'].nunique():,}"
)
print(
    "Below-target exceptions retained: "
    f"{int(chunks_df['below_minimum_target'].sum()):,}"
)

display(
    chunks_df[
        [
            "chunk_id",
            "document_id",
            "chunk_index",
            "token_count",
            "ticker",
            "company",
            "form_type",
            "filing_date",
            "report_date",
            "text",
        ]
    ].head(3)
)

Canonical chunk rows: 4,174
Unique chunk IDs: 4,174
Documents represented: 30
Below-target exceptions retained: 19


,chunk_id,document_id,chunk_index,token_count,ticker,company,form_type,filing_date,report_date,text
0,aapl_10k_000032019323000106__chunk_0000,aapl_10k_000032019323000106,0,453,AAPL,Apple Inc.,10-K,2023-11-03,2023-09-30,"aapl-20230930 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) O..."
1,aapl_10k_000032019323000106__chunk_0001,aapl_10k_000032019323000106,1,466,AAPL,Apple Inc.,10-K,2023-11-03,2023-09-30,"Employer Identification No.) One Apple Park Way Cupertino , California 95014 (Address of principal executive offices) (Zip Code) ( 408 ) 996-1010 (Registran..."
2,aapl_10k_000032019323000106__chunk_0002,aapl_10k_000032019323000106,2,360,AAPL,Apple Inc.,10-K,2023-11-03,2023-09-30,Yes ☐ No ☒ Indicate by check mark whether the Registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Ac...


In [ ]:
{
    name: str(value)
    for name, value in globals().items()
    if (
        "DIR" in name.upper()
        or "PATH" in name.upper()
        or "ARTIFACT" in name.upper()
    )
    and not name.startswith("_")
}

{'Path': "<class 'pathlib.Path'>",
 'RAW_CSV_PATH': '/content/drive/MyDrive/finance-rag-analyst/data/raw/sec_filings_raw.csv',
 'MANIFEST_CSV_PATH': '/content/drive/MyDrive/finance-rag-analyst/data/manifests/corpus_manifest_sec30_959ac4a68dbf.csv',
 'CORPUS_METADATA_PATH': '/content/drive/MyDrive/finance-rag-analyst/data/manifests/corpus_metadata_sec30_959ac4a68dbf.json',
 'REQUIRED_INPUT_PATHS': "{'raw filing CSV': PosixPath('/content/drive/MyDrive/finance-rag-analyst/data/raw/sec_filings_raw.csv'), 'frozen corpus manifest': PosixPath('/content/drive/MyDrive/finance-rag-analyst/data/manifests/corpus_manifest_sec30_959ac4a68dbf.csv'), 'corpus metadata': PosixPath('/content/drive/MyDrive/finance-rag-analyst/data/manifests/corpus_metadata_sec30_959ac4a68dbf.json')}",
 'path': '/content/drive/MyDrive/finance-rag-analyst/data/manifests/corpus_metadata_sec30_959ac4a68dbf.json'}

Cell 23 — Persist and verify the canonical chunk artifact

In [ ]:
{
    name: value
    for name, value in globals().items()
    if any(
        keyword in name.upper()
        for keyword in [
            "EMBED",
            "TOKENIZER",
            "MODEL_NAME",
            "CHUNK_TOKEN",
        ]
    )
    and isinstance(value, (str, int, float, bool))
}

{'TOKENIZER_NAME': 'BAAI/bge-small-en-v1.5', 'MAX_CHUNK_TOKENS': 480}

In [ ]:
# Cell 23 — Persist and verify the canonical chunk artifact

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/finance-rag-analyst")
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the frozen corpus identifier already embedded in the source manifest.
corpus_artifact_id = Path(MANIFEST_CSV_PATH).stem.removeprefix(
    "corpus_manifest_"
)

CHUNKS_PARQUET_PATH = (
    PROCESSED_DATA_DIR
    / f"retrieval_chunks_{corpus_artifact_id}.parquet"
)

CHUNKS_MANIFEST_PATH = (
    ARTIFACT_MANIFEST_DIR
    / f"retrieval_chunks_{corpus_artifact_id}_metadata.json"
)


def calculate_file_sha256(file_path: Path) -> str:
    """Calculate the SHA-256 checksum of a persisted artifact."""
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


# Persist without the pandas display index.
chunks_df.to_parquet(
    CHUNKS_PARQUET_PATH,
    index=False,
    engine="pyarrow",
)

chunks_file_sha256 = calculate_file_sha256(
    CHUNKS_PARQUET_PATH
)

chunks_manifest = {
    "artifact_type": "canonical_retrieval_chunks",
    "artifact_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "corpus_artifact_id": corpus_artifact_id,
    "corpus_version": str(
        chunks_df["corpus_version"].iloc[0]
    ),
    "processing_version": str(
        chunks_df["processing_version"].iloc[0]
    ),
    "source_manifest_path": str(MANIFEST_CSV_PATH),
    "source_corpus_metadata_path": str(
        CORPUS_METADATA_PATH
    ),
    "chunks_path": str(CHUNKS_PARQUET_PATH),
    "chunks_file_sha256": chunks_file_sha256,
    "chunks_file_size_bytes": (
        CHUNKS_PARQUET_PATH.stat().st_size
    ),
    "row_count": int(len(chunks_df)),
    "column_count": int(len(chunks_df.columns)),
    "document_count": int(
        chunks_df["document_id"].nunique()
    ),
    "unique_chunk_id_count": int(
        chunks_df["chunk_id"].nunique()
    ),
    "tokenizer_model": TOKENIZER_NAME,
    "maximum_chunk_tokens": int(MAX_CHUNK_TOKENS),
    "target_overlap_tokens": int(
        TARGET_OVERLAP_TOKENS
    ),
    "minimum_target_tokens": int(
        MINIMUM_TARGET_TOKENS
    ),
    "minimum_observed_tokens": int(
        chunks_df["token_count"].min()
    ),
    "maximum_observed_tokens": int(
        chunks_df["token_count"].max()
    ),
    "mean_observed_tokens": round(
        float(chunks_df["token_count"].mean()),
        2,
    ),
    "below_minimum_target_count": int(
        chunks_df["below_minimum_target"].sum()
    ),
    "columns": chunks_df.columns.tolist(),
}

with CHUNKS_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        chunks_manifest,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )
    file_handle.write("\n")


# Reload from disk and verify exact table identity.
reloaded_chunks_df = pd.read_parquet(
    CHUNKS_PARQUET_PATH,
    engine="pyarrow",
)

pd.testing.assert_frame_equal(
    chunks_df.reset_index(drop=True),
    reloaded_chunks_df.reset_index(drop=True),
    check_dtype=True,
    check_exact=True,
)

assert calculate_file_sha256(
    CHUNKS_PARQUET_PATH
) == chunks_file_sha256

assert len(reloaded_chunks_df) == 4_174
assert reloaded_chunks_df["chunk_id"].is_unique
assert reloaded_chunks_df["document_id"].nunique() == 30
assert reloaded_chunks_df["token_count"].max() <= 480
assert (
    reloaded_chunks_df["below_minimum_target"].sum()
    == 19
)

print(f"Chunk artifact: {CHUNKS_PARQUET_PATH}")
print(f"Chunk manifest: {CHUNKS_MANIFEST_PATH}")
print(f"Rows reloaded and verified: {len(reloaded_chunks_df):,}")
print(f"Artifact size: {CHUNKS_PARQUET_PATH.stat().st_size:,} bytes")
print(f"SHA-256: {chunks_file_sha256}")
print("Exact DataFrame round-trip: PASSED")

Chunk artifact: /content/drive/MyDrive/finance-rag-analyst/data/processed/retrieval_chunks_sec30_959ac4a68dbf.parquet
Chunk manifest: /content/drive/MyDrive/finance-rag-analyst/data/manifests/retrieval_chunks_sec30_959ac4a68dbf_metadata.json
Rows reloaded and verified: 4,174
Artifact size: 3,433,825 bytes
SHA-256: c41480fd37f6714c27f1c1a4a69e90634ec841cde76523d20d53559266864dd5
Exact DataFrame round-trip: PASSED




## 5. Dense Embedding Construction

The canonical retrieval chunks are encoded using
`BAAI/bge-small-en-v1.5`, the same tokenizer family used to enforce the
480-token chunk ceiling.

The embedding stage:

1. loads a frozen SentenceTransformers model;
2. encodes chunks in deterministic batches;
3. produces one embedding per canonical `chunk_id`;
4. validates shape, dtype, finiteness, and L2 normalization;
5. preserves row-order alignment between the chunk table and embedding matrix.

For BGE retrieval, document chunks are embedded as their source text.
Query-specific BGE formatting is applied later in the retrieval layer rather
than modifying the stored corpus text.

Configure and load the embedding model

In [ ]:
# Cell 25 — Configure and load the frozen embedding model

import numpy as np
import sentence_transformers
import torch

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = TOKENIZER_NAME
EMBEDDING_BATCH_SIZE = 32
EMBEDDING_NORMALIZE = True

# Use a GPU when Colab provides one; otherwise retain a valid CPU fallback.
EMBEDDING_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=EMBEDDING_DEVICE,
)

embedding_dimension = int(
    embedding_model.get_embedding_dimension()
)

model_max_sequence_length = int(
    embedding_model.max_seq_length
)

assert EMBEDDING_MODEL_NAME == "BAAI/bge-small-en-v1.5"
assert embedding_dimension == 384
assert model_max_sequence_length >= MAX_CHUNK_TOKENS
assert EMBEDDING_BATCH_SIZE > 0

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"SentenceTransformers version: {sentence_transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Embedding device: {EMBEDDING_DEVICE}")
print(f"Embedding dimension: {embedding_dimension}")
print(f"Model maximum sequence length: {model_max_sequence_length}")
print(f"Embedding batch size: {EMBEDDING_BATCH_SIZE}")
print(f"L2 normalization requested: {EMBEDDING_NORMALIZE}")

Embedding model: BAAI/bge-small-en-v1.5
SentenceTransformers version: 5.6.0
PyTorch version: 2.11.0+cpu
Embedding device: cpu
Embedding dimension: 384
Model maximum sequence length: 512
Embedding batch size: 32
L2 normalization requested: True


Encode and validate all canonical chunks

In [ ]:
# Cell 26 — Encode and validate canonical chunk embeddings

from time import perf_counter

# Encode the verified on-disk canonical artifact, preserving its row order.
embedding_source_df = reloaded_chunks_df.reset_index(drop=True)

embedding_texts = embedding_source_df["text"].tolist()
embedding_chunk_ids = embedding_source_df["chunk_id"].tolist()

assert len(embedding_texts) == 4_174
assert len(embedding_chunk_ids) == 4_174
assert len(set(embedding_chunk_ids)) == 4_174
assert all(
    isinstance(text, str) and text.strip()
    for text in embedding_texts
)

embedding_start_time = perf_counter()

chunk_embeddings = embedding_model.encode(
    embedding_texts,
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=EMBEDDING_NORMALIZE,
)

embedding_elapsed_seconds = (
    perf_counter() - embedding_start_time
)

# Standardize the persisted matrix representation.
chunk_embeddings = np.asarray(
    chunk_embeddings,
    dtype=np.float32,
    order="C",
)

expected_embedding_shape = (
    len(embedding_source_df),
    embedding_dimension,
)

assert chunk_embeddings.shape == expected_embedding_shape
assert chunk_embeddings.dtype == np.float32
assert chunk_embeddings.flags["C_CONTIGUOUS"]
assert np.isfinite(chunk_embeddings).all()

embedding_norms = np.linalg.norm(
    chunk_embeddings,
    axis=1,
)

assert np.allclose(
    embedding_norms,
    1.0,
    rtol=1e-4,
    atol=1e-5,
)

embedding_throughput = (
    len(embedding_source_df)
    / embedding_elapsed_seconds
)

embedding_matrix_size_bytes = (
    chunk_embeddings.nbytes
)

print(
    "Embedding matrix shape: "
    f"{chunk_embeddings.shape}"
)
print(f"Embedding dtype: {chunk_embeddings.dtype}")
print(
    "Finite embedding values: "
    f"{np.isfinite(chunk_embeddings).all()}"
)
print(
    "Embedding-norm range: "
    f"{embedding_norms.min():.6f}–"
    f"{embedding_norms.max():.6f}"
)
print(
    "Mean embedding norm: "
    f"{embedding_norms.mean():.6f}"
)
print(
    "Encoding time: "
    f"{embedding_elapsed_seconds:.2f} seconds"
)
print(
    "Encoding throughput: "
    f"{embedding_throughput:.2f} chunks/second"
)
print(
    "Embedding matrix size: "
    f"{embedding_matrix_size_bytes:,} bytes "
    f"({embedding_matrix_size_bytes / 1024**2:.2f} MiB)"
)
print("Chunk-to-embedding row alignment: PRESERVED")

Batches:   0%|          | 0/131 [00:00<?, ?it/s]

Embedding matrix shape: (4174, 384)
Embedding dtype: float32
Finite embedding values: True
Embedding-norm range: 1.000000–1.000000
Mean embedding norm: 1.000000
Encoding time: 2552.44 seconds
Encoding throughput: 1.64 chunks/second
Embedding matrix size: 6,411,264 bytes (6.11 MiB)
Chunk-to-embedding row alignment: PRESERVED


In [ ]:
# Cell 27 — Persist and verify embeddings and chunk-row mapping

import platform

EMBEDDING_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "data"
    / "artifacts"
    / "embeddings"
)

EMBEDDING_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EMBEDDINGS_NPY_PATH = (
    EMBEDDING_ARTIFACT_DIR
    / f"chunk_embeddings_{corpus_artifact_id}.npy"
)

EMBEDDING_MAPPING_PATH = (
    EMBEDDING_ARTIFACT_DIR
    / f"embedding_chunk_mapping_{corpus_artifact_id}.parquet"
)

EMBEDDING_MANIFEST_PATH = (
    ARTIFACT_MANIFEST_DIR
    / f"chunk_embeddings_{corpus_artifact_id}_metadata.json"
)


# Construct an explicit row-to-chunk identity mapping.
embedding_mapping_df = (
    embedding_source_df[
        [
            "chunk_id",
            "document_id",
            "chunk_index",
            "ticker",
            "form_type",
            "filing_date",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

embedding_mapping_df.insert(
    0,
    "embedding_row",
    np.arange(
        len(embedding_mapping_df),
        dtype=np.int64,
    ),
)


# Validate alignment before persistence.
assert len(embedding_mapping_df) == len(chunk_embeddings)
assert embedding_mapping_df["embedding_row"].tolist() == list(
    range(len(embedding_mapping_df))
)
assert embedding_mapping_df["chunk_id"].tolist() == embedding_chunk_ids
assert embedding_mapping_df["chunk_id"].is_unique


# Persist the matrix and its ordered identity mapping.
np.save(
    EMBEDDINGS_NPY_PATH,
    chunk_embeddings,
    allow_pickle=False,
)

embedding_mapping_df.to_parquet(
    EMBEDDING_MAPPING_PATH,
    index=False,
    engine="pyarrow",
)


# Calculate persisted-artifact fingerprints.
embeddings_file_sha256 = calculate_file_sha256(
    EMBEDDINGS_NPY_PATH
)

embedding_mapping_sha256 = calculate_file_sha256(
    EMBEDDING_MAPPING_PATH
)


embedding_manifest = {
    "artifact_type": "canonical_chunk_embeddings",
    "artifact_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "corpus_artifact_id": corpus_artifact_id,
    "source_chunks_path": str(CHUNKS_PARQUET_PATH),
    "source_chunks_file_sha256": chunks_file_sha256,
    "embeddings_path": str(EMBEDDINGS_NPY_PATH),
    "embeddings_file_sha256": embeddings_file_sha256,
    "embedding_mapping_path": str(EMBEDDING_MAPPING_PATH),
    "embedding_mapping_file_sha256": embedding_mapping_sha256,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "sentence_transformers_version": (
        sentence_transformers.__version__
    ),
    "pytorch_version": torch.__version__,
    "python_version": platform.python_version(),
    "device_used": EMBEDDING_DEVICE,
    "batch_size": int(EMBEDDING_BATCH_SIZE),
    "normalized": bool(EMBEDDING_NORMALIZE),
    "row_count": int(chunk_embeddings.shape[0]),
    "embedding_dimension": int(
        chunk_embeddings.shape[1]
    ),
    "dtype": str(chunk_embeddings.dtype),
    "matrix_nbytes": int(chunk_embeddings.nbytes),
    "encoding_elapsed_seconds": round(
        float(embedding_elapsed_seconds),
        2,
    ),
    "encoding_throughput_chunks_per_second": round(
        float(embedding_throughput),
        4,
    ),
    "minimum_embedding_norm": float(
        embedding_norms.min()
    ),
    "maximum_embedding_norm": float(
        embedding_norms.max()
    ),
    "mean_embedding_norm": float(
        embedding_norms.mean()
    ),
}

with EMBEDDING_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        embedding_manifest,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )
    file_handle.write("\n")


# Reload all artifacts from disk.
reloaded_chunk_embeddings = np.load(
    EMBEDDINGS_NPY_PATH,
    allow_pickle=False,
)

reloaded_embedding_mapping_df = pd.read_parquet(
    EMBEDDING_MAPPING_PATH,
    engine="pyarrow",
)


# Verify exact persisted identity.
assert np.array_equal(
    chunk_embeddings,
    reloaded_chunk_embeddings,
)

pd.testing.assert_frame_equal(
    embedding_mapping_df,
    reloaded_embedding_mapping_df,
    check_dtype=True,
    check_exact=True,
)

assert reloaded_chunk_embeddings.shape == (4_174, 384)
assert reloaded_chunk_embeddings.dtype == np.float32
assert np.isfinite(reloaded_chunk_embeddings).all()

assert (
    reloaded_embedding_mapping_df["embedding_row"].tolist()
    == list(range(4_174))
)

assert (
    reloaded_embedding_mapping_df["chunk_id"].tolist()
    == embedding_source_df["chunk_id"].tolist()
)

assert calculate_file_sha256(
    EMBEDDINGS_NPY_PATH
) == embeddings_file_sha256

assert calculate_file_sha256(
    EMBEDDING_MAPPING_PATH
) == embedding_mapping_sha256


print(f"Embedding artifact: {EMBEDDINGS_NPY_PATH}")
print(f"Row mapping: {EMBEDDING_MAPPING_PATH}")
print(f"Embedding manifest: {EMBEDDING_MANIFEST_PATH}")
print(
    "Reloaded embedding shape: "
    f"{reloaded_chunk_embeddings.shape}"
)
print(
    "Embedding artifact size: "
    f"{EMBEDDINGS_NPY_PATH.stat().st_size:,} bytes"
)
print(f"Embedding SHA-256: {embeddings_file_sha256}")
print(f"Mapping SHA-256: {embedding_mapping_sha256}")
print("Exact embedding round-trip: PASSED")
print("Chunk-to-embedding mapping verification: PASSED")

Embedding artifact: /content/drive/MyDrive/finance-rag-analyst/data/artifacts/embeddings/chunk_embeddings_sec30_959ac4a68dbf.npy
Row mapping: /content/drive/MyDrive/finance-rag-analyst/data/artifacts/embeddings/embedding_chunk_mapping_sec30_959ac4a68dbf.parquet
Embedding manifest: /content/drive/MyDrive/finance-rag-analyst/data/manifests/chunk_embeddings_sec30_959ac4a68dbf_metadata.json
Reloaded embedding shape: (4174, 384)
Embedding artifact size: 6,411,392 bytes
Embedding SHA-256: 255b686b41dd35ae675a31ede84d0355c49c5f9800594ab6221828ef0fdea6c5
Mapping SHA-256: 2e3720f84096df150c7e40eea2bfda6df6fadede7325f3a174897927f94488cb
Exact embedding round-trip: PASSED
Chunk-to-embedding mapping verification: PASSED



## 6. Dense FAISS Index Construction

The normalized BGE chunk embeddings are indexed using an exact
`IndexFlatIP` FAISS index.

Because each embedding has unit L2 norm, inner-product search is
mathematically equivalent to cosine-similarity search:

\[
\operatorname{cosine}(x,y)
=
\frac{x \cdot y}{\|x\|_2\|y\|_2}
=
x \cdot y
\quad \text{when} \quad
\|x\|_2=\|y\|_2=1
\]

An exact flat index is appropriate for the frozen 4,174-chunk corpus:

- it introduces no approximate-nearest-neighbor recall loss;
- it requires no training stage;
- it provides a reliable dense-retrieval baseline;
- its row identifiers map directly to the persisted embedding-row mapping.

The index is validated for dimensionality, vector count, reconstruction
accuracy, self-retrieval behavior, and persistence round-trip integrity.

In [ ]:
%pip install -q faiss-cpu

import faiss

print(f"FAISS version: {faiss.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.5 MB/s eta 0:00:00
FAISS version: 1.15.0


In [ ]:
# Cell 29 — Build and validate the exact dense FAISS index

try:
    import faiss
except ImportError as exc:
    raise ImportError(
        "FAISS is not installed. In Colab, run "
        "`%pip install -q faiss-cpu` in a separate cell, "
        "restart only if Colab requires it, and then reload the "
        "persisted embedding artifacts rather than re-encoding them."
    ) from exc


FAISS_METRIC = "inner_product"
FAISS_INDEX_TYPE = "IndexFlatIP"

# Use the verified, reloaded matrix as the index source.
faiss_source_embeddings = np.asarray(
    reloaded_chunk_embeddings,
    dtype=np.float32,
    order="C",
)

assert faiss_source_embeddings.shape == (4_174, 384)
assert faiss_source_embeddings.dtype == np.float32
assert faiss_source_embeddings.flags["C_CONTIGUOUS"]
assert np.isfinite(faiss_source_embeddings).all()

faiss_source_norms = np.linalg.norm(
    faiss_source_embeddings,
    axis=1,
)

assert np.allclose(
    faiss_source_norms,
    1.0,
    rtol=1e-4,
    atol=1e-5,
)

# IndexFlatIP performs exact inner-product search.
dense_faiss_index = faiss.IndexFlatIP(
    faiss_source_embeddings.shape[1]
)

dense_faiss_index.add(faiss_source_embeddings)

assert dense_faiss_index.is_trained
assert dense_faiss_index.d == 384
assert dense_faiss_index.ntotal == 4_174


# Verify that FAISS retains the indexed vectors accurately.
reconstruction_sample_rows = np.array(
    [0, 1, 100, 1_000, 4_173],
    dtype=np.int64,
)

reconstructed_vectors = np.vstack(
    [
        dense_faiss_index.reconstruct(int(row))
        for row in reconstruction_sample_rows
    ]
).astype(np.float32)

expected_reconstructed_vectors = (
    faiss_source_embeddings[reconstruction_sample_rows]
)

assert np.allclose(
    reconstructed_vectors,
    expected_reconstructed_vectors,
    rtol=0.0,
    atol=1e-6,
)


# Every sampled normalized vector should retrieve itself at rank 1.
SELF_TEST_TOP_K = 5

self_test_scores, self_test_indices = (
    dense_faiss_index.search(
        expected_reconstructed_vectors,
        SELF_TEST_TOP_K,
    )
)

assert self_test_scores.shape == (
    len(reconstruction_sample_rows),
    SELF_TEST_TOP_K,
)

assert self_test_indices.shape == (
    len(reconstruction_sample_rows),
    SELF_TEST_TOP_K,
)

assert np.array_equal(
    self_test_indices[:, 0],
    reconstruction_sample_rows,
)

assert np.allclose(
    self_test_scores[:, 0],
    1.0,
    rtol=1e-4,
    atol=1e-5,
)

assert (
    (self_test_indices >= 0)
    & (self_test_indices < dense_faiss_index.ntotal)
).all()


self_test_results_df = pd.DataFrame(
    {
        "query_embedding_row": reconstruction_sample_rows,
        "retrieved_embedding_row": self_test_indices[:, 0],
        "rank_1_score": self_test_scores[:, 0],
        "chunk_id": (
            reloaded_embedding_mapping_df.iloc[
                reconstruction_sample_rows
            ]["chunk_id"].tolist()
        ),
    }
)

print(f"FAISS version: {faiss.__version__}")
print(f"Index type: {FAISS_INDEX_TYPE}")
print(f"Similarity metric: {FAISS_METRIC}")
print(f"Index trained: {dense_faiss_index.is_trained}")
print(f"Index dimension: {dense_faiss_index.d}")
print(f"Indexed vectors: {dense_faiss_index.ntotal:,}")
print("Sample-vector reconstruction: PASSED")
print("Rank-1 self-retrieval: PASSED")

display(self_test_results_df)

FAISS version: 1.15.0
Index type: IndexFlatIP
Similarity metric: inner_product
Index trained: True
Index dimension: 384
Indexed vectors: 4,174
Sample-vector reconstruction: PASSED
Rank-1 self-retrieval: PASSED


,query_embedding_row,retrieved_embedding_row,rank_1_score,chunk_id
0,0,0,1.0,aapl_10k_000032019323000106__chunk_0000
1,1,1,1.0,aapl_10k_000032019323000106__chunk_0001
2,100,100,1.0,aapl_10k_000032019323000106__chunk_0100
3,1000,1000,1.0,amzn_10k_000101872426000004__chunk_0117
4,4173,4173,1.0,nvda_10q_000104581026000052__chunk_0087


In [ ]:
# Cell 30 — Persist and verify the dense FAISS index

FAISS_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "data"
    / "artifacts"
    / "faiss"
)

FAISS_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FAISS_INDEX_PATH = (
    FAISS_ARTIFACT_DIR
    / f"dense_index_{corpus_artifact_id}.faiss"
)

FAISS_MANIFEST_PATH = (
    ARTIFACT_MANIFEST_DIR
    / f"dense_index_{corpus_artifact_id}_metadata.json"
)


# Persist the exact dense index.
faiss.write_index(
    dense_faiss_index,
    str(FAISS_INDEX_PATH),
)

faiss_index_file_sha256 = calculate_file_sha256(
    FAISS_INDEX_PATH
)


# Reload the index independently from disk.
reloaded_faiss_index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

assert reloaded_faiss_index.is_trained
assert reloaded_faiss_index.d == embedding_dimension
assert reloaded_faiss_index.ntotal == len(
    reloaded_embedding_mapping_df
)

assert reloaded_faiss_index.metric_type == faiss.METRIC_INNER_PRODUCT


# Repeat the same searches against the reloaded index.
reloaded_self_test_scores, reloaded_self_test_indices = (
    reloaded_faiss_index.search(
        expected_reconstructed_vectors,
        SELF_TEST_TOP_K,
    )
)

assert np.array_equal(
    self_test_indices,
    reloaded_self_test_indices,
)

assert np.array_equal(
    self_test_scores,
    reloaded_self_test_scores,
)

assert np.array_equal(
    reloaded_self_test_indices[:, 0],
    reconstruction_sample_rows,
)

assert np.allclose(
    reloaded_self_test_scores[:, 0],
    1.0,
    rtol=1e-4,
    atol=1e-5,
)


# Confirm that reconstructed vectors also survive persistence.
reloaded_reconstructed_vectors = np.vstack(
    [
        reloaded_faiss_index.reconstruct(int(row))
        for row in reconstruction_sample_rows
    ]
).astype(np.float32)

assert np.array_equal(
    reconstructed_vectors,
    reloaded_reconstructed_vectors,
)


faiss_manifest = {
    "artifact_type": "dense_faiss_index",
    "artifact_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "corpus_artifact_id": corpus_artifact_id,
    "source_chunks_path": str(CHUNKS_PARQUET_PATH),
    "source_chunks_file_sha256": chunks_file_sha256,
    "source_embeddings_path": str(EMBEDDINGS_NPY_PATH),
    "source_embeddings_file_sha256": embeddings_file_sha256,
    "embedding_mapping_path": str(EMBEDDING_MAPPING_PATH),
    "embedding_mapping_file_sha256": embedding_mapping_sha256,
    "faiss_index_path": str(FAISS_INDEX_PATH),
    "faiss_index_file_sha256": faiss_index_file_sha256,
    "faiss_version": faiss.__version__,
    "index_type": FAISS_INDEX_TYPE,
    "metric": FAISS_METRIC,
    "metric_type": int(reloaded_faiss_index.metric_type),
    "is_exact_search": True,
    "requires_training": False,
    "is_trained": bool(reloaded_faiss_index.is_trained),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embeddings_normalized": bool(EMBEDDING_NORMALIZE),
    "dimension": int(reloaded_faiss_index.d),
    "indexed_vector_count": int(
        reloaded_faiss_index.ntotal
    ),
    "index_file_size_bytes": int(
        FAISS_INDEX_PATH.stat().st_size
    ),
    "row_identity_source": str(EMBEDDING_MAPPING_PATH),
    "validation": {
        "sample_vector_reconstruction": "passed",
        "rank_1_self_retrieval": "passed",
        "persistence_search_identity": "passed",
        "persistence_reconstruction_identity": "passed",
        "sample_rows": reconstruction_sample_rows.tolist(),
        "self_test_top_k": int(SELF_TEST_TOP_K),
    },
}

with FAISS_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        faiss_manifest,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )
    file_handle.write("\n")


# Validate the persisted files and their recorded relationships.
assert FAISS_INDEX_PATH.exists()
assert FAISS_MANIFEST_PATH.exists()

assert calculate_file_sha256(
    FAISS_INDEX_PATH
) == faiss_index_file_sha256

assert (
    reloaded_faiss_index.ntotal
    == reloaded_chunk_embeddings.shape[0]
    == len(reloaded_embedding_mapping_df)
    == len(reloaded_chunks_df)
    == 4_174
)

print(f"FAISS index: {FAISS_INDEX_PATH}")
print(f"FAISS manifest: {FAISS_MANIFEST_PATH}")
print(
    "Reloaded index type: "
    f"{type(reloaded_faiss_index).__name__}"
)
print(
    "Reloaded indexed vectors: "
    f"{reloaded_faiss_index.ntotal:,}"
)
print(
    "FAISS index size: "
    f"{FAISS_INDEX_PATH.stat().st_size:,} bytes"
)
print(f"FAISS SHA-256: {faiss_index_file_sha256}")
print("Search-result persistence identity: PASSED")
print("Vector-reconstruction persistence identity: PASSED")
print("FAISS-to-chunk row-count alignment: PASSED")

FAISS index: /content/drive/MyDrive/finance-rag-analyst/data/artifacts/faiss/dense_index_sec30_959ac4a68dbf.faiss
FAISS manifest: /content/drive/MyDrive/finance-rag-analyst/data/manifests/dense_index_sec30_959ac4a68dbf_metadata.json
Reloaded index type: IndexFlatIP
Reloaded indexed vectors: 4,174
FAISS index size: 6,411,309 bytes
FAISS SHA-256: 402f984e12a762f1e2ab1c2224cf7803d3f502fee79845896426c1b4e3802893
Search-result persistence identity: PASSED
Vector-reconstruction persistence identity: PASSED
FAISS-to-chunk row-count alignment: PASSED


In [ ]:
%pip install -q rank-bm25

## 7. Sparse BM25 Index Construction

A BM25 sparse-retrieval index is constructed over the same 4,174 canonical
chunks used by the dense FAISS index.

BM25 complements dense semantic retrieval by emphasizing exact lexical
evidence, including:

- company and product names;
- financial terminology;
- filing-specific phrases;
- numeric expressions;
- form identifiers such as `10-K` and `10-Q`.

The tokenizer applies deterministic Unicode normalization and lowercase
lexical tokenization. Hyphenated and apostrophe-containing terms are retained
as single tokens when possible.

The BM25 implementation uses `rank_bm25.BM25Okapi` with explicitly recorded
parameters. Corpus order is preserved so that every BM25 document position,
FAISS vector row, embedding row, and canonical chunk row refers to the same
`chunk_id`.

In [ ]:
# Cell 32 — Tokenize the canonical chunks and construct BM25

import importlib.metadata
import re
import unicodedata

from rank_bm25 import BM25Okapi


RANK_BM25_VERSION = importlib.metadata.version("rank-bm25")

BM25_K1 = 1.5
BM25_B = 0.75
BM25_EPSILON = 0.25

# Retain ordinary words, numbers, hyphenated terms, and apostrophes.
BM25_TOKEN_PATTERN = re.compile(
    r"[a-z0-9]+(?:[-'][a-z0-9]+)*"
)


def tokenize_for_bm25(text: str) -> list[str]:
    """Deterministically normalize and tokenize text for BM25."""
    if not isinstance(text, str):
        raise TypeError(
            "BM25 input must be a string, "
            f"received {type(text).__name__}."
        )

    normalized_text = unicodedata.normalize(
        "NFKC",
        text,
    ).lower()

    return BM25_TOKEN_PATTERN.findall(normalized_text)


# Use the same verified row order used by the embedding and FAISS artifacts.
bm25_source_df = reloaded_chunks_df.reset_index(drop=True)

assert (
    bm25_source_df["chunk_id"].tolist()
    == reloaded_embedding_mapping_df["chunk_id"].tolist()
)
assert len(bm25_source_df) == 4_174
assert bm25_source_df["chunk_id"].is_unique


bm25_tokenized_corpus = [
    tokenize_for_bm25(text)
    for text in bm25_source_df["text"]
]

assert len(bm25_tokenized_corpus) == 4_174
assert all(
    isinstance(tokens, list)
    for tokens in bm25_tokenized_corpus
)
assert all(
    len(tokens) > 0
    for tokens in bm25_tokenized_corpus
)
assert all(
    isinstance(token, str) and token
    for tokens in bm25_tokenized_corpus
    for token in tokens
)


bm25_index = BM25Okapi(
    bm25_tokenized_corpus,
    k1=BM25_K1,
    b=BM25_B,
    epsilon=BM25_EPSILON,
)


bm25_document_lengths = np.asarray(
    [len(tokens) for tokens in bm25_tokenized_corpus],
    dtype=np.int64,
)

bm25_total_tokens = int(
    bm25_document_lengths.sum()
)

bm25_vocabulary = {
    token
    for document_tokens in bm25_tokenized_corpus
    for token in document_tokens
}

bm25_vocabulary_size = len(bm25_vocabulary)


# Validate the constructed index.
assert bm25_index.corpus_size == 4_174
assert len(bm25_index.doc_len) == 4_174
assert np.array_equal(
    np.asarray(bm25_index.doc_len, dtype=np.int64),
    bm25_document_lengths,
)
assert np.isclose(
    bm25_index.avgdl,
    bm25_document_lengths.mean(),
)
assert bm25_index.k1 == BM25_K1
assert bm25_index.b == BM25_B
assert bm25_index.epsilon == BM25_EPSILON
assert bm25_vocabulary_size > 0


# Confirm that BM25 returns one finite score for every corpus row.
bm25_validation_query = tokenize_for_bm25(
    "revenue operating income risk factors"
)

bm25_validation_scores = np.asarray(
    bm25_index.get_scores(bm25_validation_query),
    dtype=np.float64,
)

assert bm25_validation_scores.shape == (4_174,)
assert np.isfinite(bm25_validation_scores).all()
assert (bm25_validation_scores >= 0).all()


BM25_VALIDATION_TOP_K = 5

bm25_validation_top_rows = np.argsort(
    -bm25_validation_scores,
    kind="stable",
)[:BM25_VALIDATION_TOP_K]

bm25_validation_results_df = pd.DataFrame(
    {
        "bm25_row": bm25_validation_top_rows,
        "bm25_score": (
            bm25_validation_scores[
                bm25_validation_top_rows
            ]
        ),
        "chunk_id": (
            bm25_source_df.iloc[
                bm25_validation_top_rows
            ]["chunk_id"].tolist()
        ),
        "ticker": (
            bm25_source_df.iloc[
                bm25_validation_top_rows
            ]["ticker"].tolist()
        ),
        "form_type": (
            bm25_source_df.iloc[
                bm25_validation_top_rows
            ]["form_type"].tolist()
        ),
    }
)


print(f"rank-bm25 version: {RANK_BM25_VERSION}")
print(f"BM25 implementation: {type(bm25_index).__name__}")
print(f"BM25 corpus documents: {bm25_index.corpus_size:,}")
print(f"BM25 total indexed tokens: {bm25_total_tokens:,}")
print(f"BM25 vocabulary size: {bm25_vocabulary_size:,}")
print(
    "BM25 document-length range: "
    f"{bm25_document_lengths.min():,}–"
    f"{bm25_document_lengths.max():,} tokens"
)
print(f"BM25 average document length: {bm25_index.avgdl:.2f}")
print(
    "BM25 parameters: "
    f"k1={bm25_index.k1}, "
    f"b={bm25_index.b}, "
    f"epsilon={bm25_index.epsilon}"
)
print("BM25-to-chunk row alignment: PRESERVED")
print("BM25 finite-score validation: PASSED")

display(bm25_validation_results_df)

rank-bm25 version: 0.2.2
BM25 implementation: BM25Okapi
BM25 corpus documents: 4,174
BM25 total indexed tokens: 1,357,568
BM25 vocabulary size: 9,459
BM25 document-length range: 9–424 tokens
BM25 average document length: 325.24
BM25 parameters: k1=1.5, b=0.75, epsilon=0.25
BM25-to-chunk row alignment: PRESERVED
BM25 finite-score validation: PASSED


,bm25_row,bm25_score,chunk_id,ticker,form_type
0,4035,8.513690,nvda_10q_000104581025000230__chunk_0050,NVDA,10-Q
1,3945,8.485430,nvda_10q_000104581025000209__chunk_0051,NVDA,10-Q
2,3792,8.481556,nvda_10k_000104581026000021__chunk_0105,NVDA,10-K
3,4129,8.470129,nvda_10q_000104581026000052__chunk_0043,NVDA,10-Q
4,2932,7.982911,msft_10q_000119312525256321__chunk_0002,MSFT,10-Q


### 7.1 BM25 Artifact Persistence

The BM25 index is represented by reproducible inputs rather than by a
version-sensitive Python pickle.

The persisted sparse-retrieval assets include:

- the ordered tokenized corpus;
- an explicit BM25-row-to-`chunk_id` mapping;
- the tokenizer specification;
- BM25 hyperparameters and package version;
- corpus statistics and cryptographic checksums.

On reload, `BM25Okapi` is reconstructed from the persisted tokens and validated
against the original index. This keeps the artifact inspectable and preserves
the row identity required for later hybrid fusion with FAISS.

In [ ]:
# Cell 34 — Persist and verify reproducible BM25 artifacts

BM25_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "data"
    / "artifacts"
    / "bm25"
)

BM25_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BM25_CORPUS_PATH = (
    BM25_ARTIFACT_DIR
    / f"bm25_tokenized_corpus_{corpus_artifact_id}.parquet"
)

BM25_MANIFEST_PATH = (
    ARTIFACT_MANIFEST_DIR
    / f"bm25_index_{corpus_artifact_id}_metadata.json"
)


# Store explicit BM25 row identity alongside each token list.
bm25_artifact_df = pd.DataFrame(
    {
        "bm25_row": np.arange(
            len(bm25_source_df),
            dtype=np.int64,
        ),
        "chunk_id": bm25_source_df["chunk_id"].tolist(),
        "tokens": bm25_tokenized_corpus,
    }
)


# Validate the artifact representation before writing.
assert len(bm25_artifact_df) == 4_174
assert bm25_artifact_df["bm25_row"].tolist() == list(
    range(4_174)
)
assert bm25_artifact_df["chunk_id"].is_unique
assert (
    bm25_artifact_df["chunk_id"].tolist()
    == reloaded_embedding_mapping_df["chunk_id"].tolist()
)
assert all(
    isinstance(tokens, list) and len(tokens) > 0
    for tokens in bm25_artifact_df["tokens"]
)


# Persist the ordered tokenized corpus.
bm25_artifact_df.to_parquet(
    BM25_CORPUS_PATH,
    index=False,
    engine="pyarrow",
)

bm25_corpus_file_sha256 = calculate_file_sha256(
    BM25_CORPUS_PATH
)


bm25_manifest = {
    "artifact_type": "reproducible_bm25_inputs",
    "artifact_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "corpus_artifact_id": corpus_artifact_id,
    "source_chunks_path": str(CHUNKS_PARQUET_PATH),
    "source_chunks_file_sha256": chunks_file_sha256,
    "bm25_corpus_path": str(BM25_CORPUS_PATH),
    "bm25_corpus_file_sha256": bm25_corpus_file_sha256,
    "rank_bm25_version": RANK_BM25_VERSION,
    "implementation": type(bm25_index).__name__,
    "serialization_strategy": (
        "persist_tokenized_corpus_and_reconstruct_index"
    ),
    "pickle_used": False,
    "tokenizer": {
        "unicode_normalization": "NFKC",
        "lowercase": True,
        "regex_pattern": BM25_TOKEN_PATTERN.pattern,
        "regex_flags": int(BM25_TOKEN_PATTERN.flags),
        "description": (
            "Alphanumeric tokens retaining internal hyphens "
            "and apostrophes."
        ),
    },
    "parameters": {
        "k1": float(BM25_K1),
        "b": float(BM25_B),
        "epsilon": float(BM25_EPSILON),
    },
    "corpus_document_count": int(
        bm25_index.corpus_size
    ),
    "total_indexed_tokens": int(
        bm25_total_tokens
    ),
    "vocabulary_size": int(
        bm25_vocabulary_size
    ),
    "minimum_document_length_tokens": int(
        bm25_document_lengths.min()
    ),
    "maximum_document_length_tokens": int(
        bm25_document_lengths.max()
    ),
    "average_document_length_tokens": float(
        bm25_index.avgdl
    ),
    "row_identity": {
        "row_column": "bm25_row",
        "chunk_id_column": "chunk_id",
        "aligned_with_chunk_rows": True,
        "aligned_with_embedding_rows": True,
        "aligned_with_faiss_rows": True,
    },
    "validation_query": {
        "text": "revenue operating income risk factors",
        "tokens": bm25_validation_query,
    },
}


with BM25_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        bm25_manifest,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )
    file_handle.write("\n")


# Reload the persisted representation independently.
reloaded_bm25_artifact_df = pd.read_parquet(
    BM25_CORPUS_PATH,
    engine="pyarrow",
)

# PyArrow may reload list columns as NumPy arrays.
reloaded_bm25_tokenized_corpus = [
    list(tokens)
    for tokens in reloaded_bm25_artifact_df["tokens"]
]


# Verify the exact row identities and tokens.
assert (
    reloaded_bm25_artifact_df["bm25_row"].tolist()
    == list(range(4_174))
)

assert (
    reloaded_bm25_artifact_df["chunk_id"].tolist()
    == bm25_source_df["chunk_id"].tolist()
)

assert (
    reloaded_bm25_tokenized_corpus
    == bm25_tokenized_corpus
)

assert calculate_file_sha256(
    BM25_CORPUS_PATH
) == bm25_corpus_file_sha256


# Reconstruct BM25 exclusively from the persisted token corpus.
reloaded_bm25_index = BM25Okapi(
    reloaded_bm25_tokenized_corpus,
    k1=BM25_K1,
    b=BM25_B,
    epsilon=BM25_EPSILON,
)

assert reloaded_bm25_index.corpus_size == 4_174
assert reloaded_bm25_index.k1 == BM25_K1
assert reloaded_bm25_index.b == BM25_B
assert reloaded_bm25_index.epsilon == BM25_EPSILON

assert np.array_equal(
    np.asarray(
        reloaded_bm25_index.doc_len,
        dtype=np.int64,
    ),
    bm25_document_lengths,
)

assert np.isclose(
    reloaded_bm25_index.avgdl,
    bm25_index.avgdl,
)


# Confirm that reconstruction preserves scoring and ranking.
reloaded_bm25_validation_scores = np.asarray(
    reloaded_bm25_index.get_scores(
        bm25_validation_query
    ),
    dtype=np.float64,
)

assert np.array_equal(
    bm25_validation_scores,
    reloaded_bm25_validation_scores,
)

reloaded_bm25_top_rows = np.argsort(
    -reloaded_bm25_validation_scores,
    kind="stable",
)[:BM25_VALIDATION_TOP_K]

assert np.array_equal(
    bm25_validation_top_rows,
    reloaded_bm25_top_rows,
)


# Confirm all retrieval representations share one row identity.
assert (
    reloaded_bm25_artifact_df["chunk_id"].tolist()
    == reloaded_embedding_mapping_df["chunk_id"].tolist()
    == reloaded_chunks_df["chunk_id"].tolist()
)

assert (
    reloaded_faiss_index.ntotal
    == reloaded_bm25_index.corpus_size
    == len(reloaded_bm25_artifact_df)
    == 4_174
)


print(f"BM25 token corpus: {BM25_CORPUS_PATH}")
print(f"BM25 manifest: {BM25_MANIFEST_PATH}")
print(
    "Reloaded BM25 documents: "
    f"{reloaded_bm25_index.corpus_size:,}"
)
print(
    "BM25 artifact size: "
    f"{BM25_CORPUS_PATH.stat().st_size:,} bytes"
)
print(f"BM25 corpus SHA-256: {bm25_corpus_file_sha256}")
print("Exact token-corpus round-trip: PASSED")
print("BM25 score reconstruction identity: PASSED")
print("BM25 ranking reconstruction identity: PASSED")
print("BM25/FAISS/chunk row alignment: PASSED")
print("Pickle-free BM25 persistence: PASSED")

BM25 token corpus: /content/drive/MyDrive/finance-rag-analyst/data/artifacts/bm25/bm25_tokenized_corpus_sec30_959ac4a68dbf.parquet
BM25 manifest: /content/drive/MyDrive/finance-rag-analyst/data/manifests/bm25_index_sec30_959ac4a68dbf_metadata.json
Reloaded BM25 documents: 4,174
BM25 artifact size: 2,340,074 bytes
BM25 corpus SHA-256: d5e7bc62973f5e95d1888da6020ab841c077d6ac0f0acf29f5a8e69b300dbd6d
Exact token-corpus round-trip: PASSED
BM25 score reconstruction identity: PASSED
BM25 ranking reconstruction identity: PASSED
BM25/FAISS/chunk row alignment: PASSED
Pickle-free BM25 persistence: PASSED


## 8. Final Retrieval-Artifact Audit

The final audit treats the canonical chunk table, embedding matrix, embedding-row
mapping, dense FAISS index, and sparse BM25 corpus as one versioned retrieval
bundle.

The audit verifies:

- identical row counts across every retrieval representation;
- identical ordered `chunk_id` mappings;
- expected embedding dimensionality and normalization;
- valid FAISS index type, metric, and vector count;
- deterministic BM25 reconstruction;
- current SHA-256 fingerprints for every persisted artifact;
- manifest-to-file checksum consistency.

A bundle manifest records the exact artifact set consumed by downstream retrieval
evaluation. Semantic retrieval quality is not evaluated here; that is performed
in Notebook 3 using independently labeled financial questions.

In [ ]:
# Cell 36 — Final cross-artifact audit and retrieval bundle manifest

RETRIEVAL_BUNDLE_MANIFEST_PATH = (
    ARTIFACT_MANIFEST_DIR
    / f"retrieval_bundle_{corpus_artifact_id}_metadata.json"
)


# Reload component manifests independently.
with EMBEDDING_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    audited_embedding_manifest = json.load(file_handle)

with FAISS_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    audited_faiss_manifest = json.load(file_handle)

with BM25_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    audited_bm25_manifest = json.load(file_handle)


# Recalculate current artifact fingerprints.
audited_chunks_sha256 = calculate_file_sha256(
    CHUNKS_PARQUET_PATH
)

audited_embeddings_sha256 = calculate_file_sha256(
    EMBEDDINGS_NPY_PATH
)

audited_embedding_mapping_sha256 = calculate_file_sha256(
    EMBEDDING_MAPPING_PATH
)

audited_faiss_sha256 = calculate_file_sha256(
    FAISS_INDEX_PATH
)

audited_bm25_sha256 = calculate_file_sha256(
    BM25_CORPUS_PATH
)


# Confirm the component manifests still describe the files on disk.
assert audited_chunks_sha256 == chunks_file_sha256

assert (
    audited_embedding_manifest["source_chunks_file_sha256"]
    == audited_chunks_sha256
)

assert (
    audited_embedding_manifest["embeddings_file_sha256"]
    == audited_embeddings_sha256
)

assert (
    audited_embedding_manifest[
        "embedding_mapping_file_sha256"
    ]
    == audited_embedding_mapping_sha256
)

assert (
    audited_faiss_manifest["source_chunks_file_sha256"]
    == audited_chunks_sha256
)

assert (
    audited_faiss_manifest["source_embeddings_file_sha256"]
    == audited_embeddings_sha256
)

assert (
    audited_faiss_manifest[
        "embedding_mapping_file_sha256"
    ]
    == audited_embedding_mapping_sha256
)

assert (
    audited_faiss_manifest["faiss_index_file_sha256"]
    == audited_faiss_sha256
)

assert (
    audited_bm25_manifest["source_chunks_file_sha256"]
    == audited_chunks_sha256
)

assert (
    audited_bm25_manifest["bm25_corpus_file_sha256"]
    == audited_bm25_sha256
)


# Confirm all manifests belong to the same frozen corpus.
assert (
    audited_embedding_manifest["corpus_artifact_id"]
    == audited_faiss_manifest["corpus_artifact_id"]
    == audited_bm25_manifest["corpus_artifact_id"]
    == corpus_artifact_id
)


# Confirm one shared ordered chunk identity.
canonical_chunk_ids = reloaded_chunks_df[
    "chunk_id"
].tolist()

assert canonical_chunk_ids == reloaded_embedding_mapping_df[
    "chunk_id"
].tolist()

assert canonical_chunk_ids == reloaded_bm25_artifact_df[
    "chunk_id"
].tolist()

assert len(canonical_chunk_ids) == len(set(canonical_chunk_ids))


# Confirm structural agreement across retrieval representations.
retrieval_row_count = len(canonical_chunk_ids)

assert retrieval_row_count == 4_174

assert reloaded_chunk_embeddings.shape == (
    retrieval_row_count,
    384,
)

assert reloaded_faiss_index.ntotal == retrieval_row_count
assert reloaded_faiss_index.d == 384

assert (
    reloaded_faiss_index.metric_type
    == faiss.METRIC_INNER_PRODUCT
)

assert (
    reloaded_bm25_index.corpus_size
    == retrieval_row_count
)

assert len(
    reloaded_bm25_tokenized_corpus
) == retrieval_row_count

assert np.isfinite(reloaded_chunk_embeddings).all()

final_embedding_norms = np.linalg.norm(
    reloaded_chunk_embeddings,
    axis=1,
)

assert np.allclose(
    final_embedding_norms,
    1.0,
    rtol=1e-4,
    atol=1e-5,
)


# Create one authoritative downstream bundle description.
retrieval_bundle_manifest = {
    "artifact_type": "retrieval_artifact_bundle",
    "artifact_version": "1.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "corpus_artifact_id": corpus_artifact_id,
    "row_count": int(retrieval_row_count),
    "shared_row_identity": "chunk_id",
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(
        reloaded_chunk_embeddings.shape[1]
    ),
    "embeddings_normalized": True,
    "dense_index": {
        "implementation": "faiss.IndexFlatIP",
        "metric": "inner_product",
        "exact_search": True,
    },
    "sparse_index": {
        "implementation": "rank_bm25.BM25Okapi",
        "k1": float(BM25_K1),
        "b": float(BM25_B),
        "epsilon": float(BM25_EPSILON),
        "pickle_used": False,
    },
    "artifacts": {
        "chunks": {
            "path": str(CHUNKS_PARQUET_PATH),
            "sha256": audited_chunks_sha256,
        },
        "embeddings": {
            "path": str(EMBEDDINGS_NPY_PATH),
            "sha256": audited_embeddings_sha256,
        },
        "embedding_mapping": {
            "path": str(EMBEDDING_MAPPING_PATH),
            "sha256": audited_embedding_mapping_sha256,
        },
        "faiss_index": {
            "path": str(FAISS_INDEX_PATH),
            "sha256": audited_faiss_sha256,
        },
        "bm25_tokenized_corpus": {
            "path": str(BM25_CORPUS_PATH),
            "sha256": audited_bm25_sha256,
        },
    },
    "component_manifests": {
        "embeddings": str(EMBEDDING_MANIFEST_PATH),
        "faiss": str(FAISS_MANIFEST_PATH),
        "bm25": str(BM25_MANIFEST_PATH),
    },
    "validation": {
        "component_checksum_integrity": "passed",
        "manifest_checksum_consistency": "passed",
        "ordered_chunk_identity": "passed",
        "row_count_alignment": "passed",
        "embedding_validation": "passed",
        "faiss_validation": "passed",
        "bm25_reconstruction_validation": "passed",
    },
    "evaluation_scope": (
        "Retrieval quality is evaluated separately in Notebook 3."
    ),
}


with RETRIEVAL_BUNDLE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        retrieval_bundle_manifest,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )
    file_handle.write("\n")


# Reload and verify the final bundle manifest.
with RETRIEVAL_BUNDLE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    reloaded_retrieval_bundle_manifest = json.load(
        file_handle
    )

assert (
    reloaded_retrieval_bundle_manifest[
        "corpus_artifact_id"
    ]
    == corpus_artifact_id
)

assert (
    reloaded_retrieval_bundle_manifest["row_count"]
    == 4_174
)

assert (
    reloaded_retrieval_bundle_manifest[
        "artifacts"
    ]["chunks"]["sha256"]
    == audited_chunks_sha256
)

assert (
    reloaded_retrieval_bundle_manifest[
        "artifacts"
    ]["embeddings"]["sha256"]
    == audited_embeddings_sha256
)

assert (
    reloaded_retrieval_bundle_manifest[
        "artifacts"
    ]["faiss_index"]["sha256"]
    == audited_faiss_sha256
)

assert (
    reloaded_retrieval_bundle_manifest[
        "artifacts"
    ]["bm25_tokenized_corpus"]["sha256"]
    == audited_bm25_sha256
)


print(
    "Retrieval bundle manifest: "
    f"{RETRIEVAL_BUNDLE_MANIFEST_PATH}"
)
print(f"Corpus artifact ID: {corpus_artifact_id}")
print(f"Canonical retrieval rows: {retrieval_row_count:,}")
print(
    "Embedding matrix: "
    f"{reloaded_chunk_embeddings.shape}"
)
print(
    "FAISS indexed vectors: "
    f"{reloaded_faiss_index.ntotal:,}"
)
print(
    "BM25 indexed documents: "
    f"{reloaded_bm25_index.corpus_size:,}"
)
print("Component checksum integrity: PASSED")
print("Manifest checksum consistency: PASSED")
print("Ordered chunk identity audit: PASSED")
print("Cross-artifact row alignment: PASSED")
print("Notebook 2 retrieval artifacts: READY")

Retrieval bundle manifest: /content/drive/MyDrive/finance-rag-analyst/data/manifests/retrieval_bundle_sec30_959ac4a68dbf_metadata.json
Corpus artifact ID: sec30_959ac4a68dbf
Canonical retrieval rows: 4,174
Embedding matrix: (4174, 384)
FAISS indexed vectors: 4,174
BM25 indexed documents: 4,174
Component checksum integrity: PASSED
Manifest checksum consistency: PASSED
Ordered chunk identity audit: PASSED
Cross-artifact row alignment: PASSED
Notebook 2 retrieval artifacts: READY


In [ ]:
# Final cell — Notebook 2 engineering metrics summary

notebook_2_metrics = {
    "corpus_artifact_id": corpus_artifact_id,
    "canonical_chunks": int(len(reloaded_chunks_df)),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(reloaded_chunk_embeddings.shape[1]),
    "embedding_dtype": str(reloaded_chunk_embeddings.dtype),
    "embeddings_normalized": bool(
        np.allclose(
            np.linalg.norm(reloaded_chunk_embeddings, axis=1),
            1.0,
            rtol=1e-4,
            atol=1e-5,
        )
    ),
    "dense_index": type(reloaded_faiss_index).__name__,
    "dense_similarity": "inner_product",
    "faiss_vectors": int(reloaded_faiss_index.ntotal),
    "faiss_index_size_mib": round(
        FAISS_INDEX_PATH.stat().st_size / (1024 ** 2),
        2,
    ),
    "sparse_index": type(reloaded_bm25_index).__name__,
    "bm25_documents": int(reloaded_bm25_index.corpus_size),
    "bm25_total_tokens": int(bm25_total_tokens),
    "bm25_vocabulary_size": int(bm25_vocabulary_size),
    "bm25_average_document_tokens": round(
        float(reloaded_bm25_index.avgdl),
        2,
    ),
    "bm25_artifact_size_mib": round(
        BM25_CORPUS_PATH.stat().st_size / (1024 ** 2),
        2,
    ),
    "cross_artifact_alignment": "PASSED",
    "checksum_integrity": "PASSED",
    "retrieval_quality_evaluation": "Deferred to Notebook 3",
}

notebook_2_metrics_df = pd.DataFrame(
    {
        "metric": notebook_2_metrics.keys(),
        "value": notebook_2_metrics.values(),
    }
)

display(notebook_2_metrics_df)

print("\nNOTEBOOK 2 — FINAL ENGINEERING SUMMARY")
print(f"Canonical chunks: {len(reloaded_chunks_df):,}")
print(
    "Dense embeddings: "
    f"{reloaded_chunk_embeddings.shape[0]:,} × "
    f"{reloaded_chunk_embeddings.shape[1]}"
)
print(
    "FAISS index: "
    f"{type(reloaded_faiss_index).__name____ if False else type(reloaded_faiss_index).__name__}, "
    f"{reloaded_faiss_index.ntotal:,} vectors"
)
print(
    "BM25 index: "
    f"{reloaded_bm25_index.corpus_size:,} documents, "
    f"{bm25_total_tokens:,} tokens, "
    f"{bm25_vocabulary_size:,} unique terms"
)
print("Artifact integrity and ordered row alignment: PASSED")
print("Retrieval-quality metrics: TO BE MEASURED IN NOTEBOOK 3")

,metric,value
0,corpus_artifact_id,sec30_959ac4a68dbf
1,canonical_chunks,4174
2,embedding_model,BAAI/bge-small-en-v1.5
3,embedding_dimension,384
4,embedding_dtype,float32
5,embeddings_normalized,True
6,dense_index,IndexFlatIP
7,dense_similarity,inner_product
8,faiss_vectors,4174
9,faiss_index_size_mib,6.11



NOTEBOOK 2 — FINAL ENGINEERING SUMMARY
Canonical chunks: 4,174
Dense embeddings: 4,174 × 384
FAISS index: IndexFlatIP, 4,174 vectors
BM25 index: 4,174 documents, 1,357,568 tokens, 9,459 unique terms
Artifact integrity and ordered row alignment: PASSED
Retrieval-quality metrics: TO BE MEASURED IN NOTEBOOK 3
